# 多输入多输出通道

In [1]:
import torch
from d2l import torch as d2l

In [2]:
""" 多输入单输出 """
def corr2d_multi_in(X, K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))
    
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

print(X)
print(K)    
corr2d_multi_in(X, K)

tensor([[[0., 1., 2.],
         [3., 4., 5.],
         [6., 7., 8.]],

        [[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]]])
tensor([[[0., 1.],
         [2., 3.]],

        [[1., 2.],
         [3., 4.]]])


tensor([[ 56.,  72.],
        [104., 120.]])

In [3]:
""" 多输入多输出 """
def corr2d_multi_in_out(X, K):
    # 迭代“K”的第0个维度，每次都对输入“X”执行互相关运算。
    # 最后将所有结果都叠加在一起
    return torch.stack([corr2d_multi_in(X, k) for k in K], dim=0)

K = torch.stack((K, K + 1, K + 2), dim=0)   
# for k in K:
#     print(corr2d_multi_in(X, k))

corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

In [4]:
""" 1*1卷积 """
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # 全连接层中的矩阵乘法
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
print(Y1)
print(Y2)
print(torch.abs(Y1 - Y2).sum() < 1e-6)

tensor([[[-0.8728, -2.6904,  1.6629],
         [ 0.2329,  0.2800, -4.1220],
         [ 1.4074, -0.4275,  0.4306]],

        [[-1.7072, -1.4118,  0.1340],
         [ 0.2716,  0.8382, -1.5973],
         [-0.6550, -0.0621,  0.0611]]])
tensor([[[-0.8728, -2.6904,  1.6629],
         [ 0.2329,  0.2800, -4.1220],
         [ 1.4074, -0.4275,  0.4306]],

        [[-1.7072, -1.4118,  0.1340],
         [ 0.2716,  0.8382, -1.5973],
         [-0.6550, -0.0621,  0.0611]]])
tensor(True)
